In [1]:
import os
import gc
from typing import Tuple

import torch
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments

import warnings
warnings.filterwarnings('ignore')

2025-09-13 06:33:08.810129: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-13 06:33:08.998071: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757745189.071902      70 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757745189.092288      70 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-13 06:33:09.272789: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
IS_KAGGLE = os.path.exists("/kaggle/input")
TRAIN_DATA = "/kaggle/input/jigsaw-agile-community-rules/train.csv" if IS_KAGGLE else "data/train.csv"
TEST_DATA = "/kaggle/input/jigsaw-agile-community-rules/test.csv" if IS_KAGGLE else "data/test.csv"

In [3]:
def load_model() -> Tuple[DistilBertTokenizer, DistilBertForSequenceClassification]:
    model_name = "distilbert-base-uncased"
    if IS_KAGGLE:
        model_path = f"/kaggle/input/transformers/{model_name}"
    else:
        model_path = "distilbert-base-uncased"
    
    tokenizer = DistilBertTokenizer.from_pretrained(model_path)
    model = DistilBertForSequenceClassification.from_pretrained(model_path, num_labels=2)

    return (tokenizer, model)

In [4]:
# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text_pair"], truncation=True)

In [19]:
## Data Cleaning 
import nltk , emoji , re 
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer =  WordNetLemmatizer()

def clean_and_lemmatize(text: str) -> str:
    # 1. Convert emojis to text (keep them as tokens)
    text = emoji.demojize(text, delimiters=(" ", " "))
    
    # 2. Replace URLs with a placeholder
    text = re.sub(r'http\S+|www\.\S+', ' <URL> ', text)
    
    # 3. Replace mentions and hashtags (optional, if present in dataset)
    text = re.sub(r'@\w+', ' <USER> ', text)
    text = re.sub(r'#\w+', ' <HASHTAG> ', text)
    
    # 4. Normalize case
    text = text.lower()
    
    # 5. Keep only words and placeholders (remove other punctuation)
    text = re.sub(r'[^a-zA-Z0-9<> ]', ' ', text)
    
    # 6. Tokenize
#     tokens = word_tokenize(text)
    
    # 7. Lemmatize each token
#     lemmatized = [lemmatizer.lemmatize(token) for token in tokens if token.strip()]
    
    return text

In [11]:
def load_train_data(tokenizer: DistilBertTokenizer) -> Tuple[Dataset, Dataset]:
    # Train Data Load
    data_path = TRAIN_DATA
    df = pd.read_csv(data_path)

    # Data Process
    expanded_data = []
  
    for _, row in df.iterrows():
        # preprocess
        body = clean_and_lemmatize(row['body'])
        rule = clean_and_lemmatize(row['rule'])
        
        # Original example
        original_text = body + tokenizer.sep_token + rule + tokenizer.sep_token
        expanded_data.append({
            'text_pair': original_text,
            'labels': row['rule_violation'],
        })
        
        # Positive examples (rule_violation = 1)
        for i in range(1, 3):
            if pd.notna(row[f'positive_example_{i}']) and row[f'positive_example_{i}'].strip():
                pos_example = row[f'positive_example_{i}'] + tokenizer.sep_token + row['rule'] + tokenizer.sep_token
                expanded_data.append({'text_pair': pos_example, 'labels': 1})
        
        # Negative examples (rule_violation = 0)
        for i in range(1, 3):
            if pd.notna(row[f'negative_example_{i}']) and row[f'negative_example_{i}'].strip():
                neg_example = row[f'negative_example_{i}'] + tokenizer.sep_token + row['rule'] + tokenizer.sep_token
                expanded_data.append({'text_pair': neg_example, 'labels': 0})

    # Test Data도 함께 로드하여 positive/negative example 활용
    test_df = pd.read_csv(TEST_DATA)
    
    for _, row in test_df.iterrows():
        # Test data의 positive examples (rule_violation = 1로 가정)
        for i in range(1, 3):
            if pd.notna(row[f'positive_example_{i}']) and row[f'positive_example_{i}'].strip():
                pos_example = row[f'positive_example_{i}'] + tokenizer.sep_token + row['rule'] + tokenizer.sep_token
                expanded_data.append({'text_pair': pos_example, 'labels': 1})
        
        # Test data의 negative examples (rule_violation = 0으로 가정)
        for i in range(1, 3):
            if pd.notna(row[f'negative_example_{i}']) and row[f'negative_example_{i}'].strip():
                neg_example = row[f'negative_example_{i}'] + tokenizer.sep_token + row['rule'] + tokenizer.sep_token
                expanded_data.append({'text_pair': neg_example, 'labels': 0})

    df = pd.DataFrame(expanded_data)

    # Data split
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    tokenized_train = train_dataset.map(tokenize_function, batched=True)
    tokenized_val = val_dataset.map(tokenize_function, batched=True)

    return (tokenized_train, tokenized_val)

In [8]:
def compute_metrics(eval_pred):
    """
    평가 시 사용할 메트릭을 계산합니다.
    
    Args:
        eval_pred: (predictions, labels) 튜플
        
    Returns:
        dict: 계산된 메트릭들
    """
    predictions, labels = eval_pred
    
    # 예측 확률 계산 (softmax 적용)
    probabilities = torch.nn.functional.softmax(torch.from_numpy(predictions), dim=1)
    
    # 각 Column별 AUC 계산
    auc_scores = {}
    # TODO
    
    # 전체 AUC (클래스 1에 대한)
    try:
        overall_auc = roc_auc_score(labels, probabilities[:, 1])
        auc_scores['overall_auc'] = overall_auc
    except ValueError:
        auc_scores['overall_auc'] = 0.0
    
    return auc_scores

In [7]:
# データバランシング
from sklearn.utils.class_weight import compute_class_weight

def get_class_weights(labels) -> dict:
    """クラス重みを計算"""
    class_weights = compute_class_weight(
        'balanced', 
        classes=np.unique(labels), 
        y=labels
    )
    return dict(zip(np.unique(labels), class_weights))

In [21]:
def train_model(model: DistilBertForSequenceClassification,
                tokenized_train: Dataset, 
                tokenized_val: Dataset, 
                tokenizer: DistilBertTokenizer) -> Trainer:
    """
    모델을 훈련합니다.
    
    Args:
        model: DistilBERT 모델
        tokenized_train: 토크나이징된 훈련 데이터셋
        tokenized_val: 토크나이징된 검증 데이터셋
        tokenizer: DistilBERT 토크나이저
        
    Returns:
        Trainer: 훈련된 트레이너 객체
    """
    labels = [example['labels'] for example in tokenized_train]
    class_weights = get_class_weights(labels)
    
    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=15,              # 훈련 에포크 수
        per_device_train_batch_size=8,   # GPU당 훈련 배치 크기
        per_device_eval_batch_size=8,    # GPU당 평가 배치 크기
        # early_stopping_patience=3,
        # early_stopping_threshold=0.001,
#         class_weight=class_weights,
        learning_rate=2e-5,
        warmup_steps=500,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        report_to="none",                # 로깅 비활성화
        metric_for_best_model="overall_auc",  # 최고 모델 선택 기준
        greater_is_better=True,          # AUC는 높을수록 좋음
        load_best_model_at_end=True      # 훈련 끝에 최고 모델 로드
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,  # 여기에 추가
    )

    print("Training started...")
    trainer.train()
    print("Training finished.")

    return trainer

In [13]:
def load_test_data(tokenizer: DistilBertTokenizer) -> Tuple[Dataset, pd.DataFrame]:
    test_df = pd.read_csv(TEST_DATA)
    test_df['text_pair'] = test_df['body'] + tokenizer.sep_token + test_df['rule'] + tokenizer.sep_token
    test_dataset = Dataset.from_pandas(test_df)
    tokenized_test = test_dataset.map(tokenize_function, batched=True)
    tokenized_test = tokenized_test.remove_columns(["body", "rule", "subreddit", "positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"])

    return (tokenized_test, test_df)

In [15]:
def predict_model(model: DistilBertForSequenceClassification, trainer: Trainer, tokenized_test: Dataset, test_df: pd.DataFrame):
    # Predict
    model.eval()

    # 예측 수행
    predictions = trainer.predict(tokenized_test)
    probabilities = torch.nn.functional.softmax(torch.from_numpy(predictions.predictions), dim=1)[:, 1].numpy()

    # 제출 파일 생성
    submission_df = pd.DataFrame({'row_id': test_df['row_id'], 'rule_violation': probabilities})
    submission_df.to_csv('submission.csv', index=False)

    print("Submission file created successfully.")

In [22]:
# Run
tokenizer, model = load_model()
tokenized_train, tokenized_val = load_train_data(tokenizer)
tokenized_test, test_df = load_test_data(tokenizer)
trainer = train_model(model, tokenized_train, tokenized_val, tokenizer)
predict_model(model, trainer, tokenized_test, test_df)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/8148 [00:00<?, ? examples/s]

Map:   0%|          | 0/2037 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Training started...


Epoch,Training Loss,Validation Loss,Overall Auc
1,0.231200,0.173994,0.991933
2,0.065300,0.147269,0.995976
3,0.138300,0.176462,0.995583
4,0.000400,0.212807,0.994641
5,0.064900,0.223443,0.995086
6,0.000100,0.211355,0.995327
7,0.000100,0.263409,0.994409
8,0.016500,0.268067,0.993115
9,0.000000,0.290348,0.993378
10,0.000000,0.253474,0.995344


Training finished.


Submission file created successfully.


In [13]:
del model
del tokenizer
del trainer
torch.cuda.empty_cache()
gc.collect()

758

In [23]:
## model dataset create
## https://medium.com/@bingqian/how-to-use-a-hugging-face-model-without-internet-access-bfba1267416c

def load_model() -> Tuple[DistilBertTokenizer, DistilBertForSequenceClassification]:
    model_name = "distilbert-base-uncased"
    if IS_KAGGLE:
        model_path = f"/kaggle/input/transformers/{model_name}"
    else:
        model_path = "distilbert-base-uncased"
    
    tokenizer = DistilBertTokenizer.from_pretrained(model_path)
    model = DistilBertForSequenceClassification.from_pretrained(model_path, num_labels=2)

    return (tokenizer, model)

save_path = "./distilbert-base-uncased"

tokenizer, model = load_model()

NameError: name 'tokeni' is not defined

In [30]:
import os
import shutil
from transformers import DistilBertModel, DistilBertConfig

model_name = "distilbert-base-uncased"
save_path = "./distilbert-base-uncased" 

# 既存のディレクトリを削除
if os.path.exists(save_path):
    shutil.rmtree(save_path)

# 1. まずbase modelを保存
print("Downloading base model...")
base_model = DistilBertModel.from_pretrained(model_name)
base_model.save_pretrained(save_path, safe_serialization=False)

# 2. configを更新して分類用に設定
config = DistilBertConfig.from_pretrained(model_name)
config.num_labels = 2
config.save_pretrained(save_path)

# 3. トークナイザーを保存
print("Downloading tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained(model_name)
tokenizer.save_pretrained(save_path)

print("Download and save completed!")

TypeError: stat: path should be string, bytes, os.PathLike or integer, not NoneType

In [31]:
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

TypeError: stat: path should be string, bytes, os.PathLike or integer, not NoneType

In [28]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [35]:
from huggingface_hub import hf_hub_download
import os

model_name = "distilbert-base-uncased"
save_path = "./distilbert-base-uncased"

# 既存のディレクトリを削除
if os.path.exists(save_path):
    shutil.rmtree(save_path)
# 新規作成
os.makedirs(save_path, exist_ok=True)

# 必要なファイルを個別にダウンロード
files_to_download = [
    "config.json",
#     "flax_model.msgpack",
    "model.safetensors",
    "pytorch_model.bin", 
#     "rust_model.ot",
#     "tf_model.h5"
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt"
]

In [36]:
for filename in files_to_download:
    try:
        print(f"Downloading {filename}...")
        file_path = hf_hub_download(
            repo_id=model_name,
            filename=filename,
            local_dir=save_path,
            local_dir_use_symlinks=False
        )
        print(f"Downloaded: {filename}")
    except Exception as e:
        print(f"Failed to download {filename}: {e}")

print("Manual download completed!")

Downloaded: config.json
Downloaded: model.safetensors


pytorch_model.bin:   0%|          | 0.00/268M [00:00<?, ?B/s]

Downloaded: pytorch_model.bin
Downloaded: tokenizer.json
Downloaded: tokenizer_config.json
Downloaded: vocab.txt
Manual download completed!


In [37]:
!ls ./distilbert-base-uncased

config.json	   pytorch_model.bin	  tokenizer.json
model.safetensors  tokenizer_config.json  vocab.txt


In [38]:
import zipfile

# zipファイルを作成
with zipfile.ZipFile('distilbert-base-uncased.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(save_path):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, os.path.dirname(save_path))
            zipf.write(file_path, arcname)

print("ZIP file created: distilbert-base-uncased.zip")

ZIP file created: distilbert-base-uncased.zip
